# Investigação dos dados e consultas SQL

Este notebook lê o arquivo com os dados integrados e realiza consultas SQL para responder perguntas sobre o conjunto de dados para testar as hipóteses definidas na etapa de 'Preparação e integração dos dados'.

A primeira consulta realizada foi "Quantas interrupções ocorreram nos últimos 5 anos? Quais estados da Região Sul tiveram uma proporção maior de interrupções?". Essa consulta visa 

In [ ]:
from pathlib import Path

import duckdb

interruption_file = Path("../data/raw/ONS/INTERRUPCAO_CARGA.csv")
processed_file = Path("../data/processed/ons_inmet_sul.parquet")

INICIO = "2021-01-01"
FIM = "2025-12-31"

In [ ]:
periodo = f"""
    SELECT cod_perturbacao AS perturbacao, id_estado AS estado
    FROM read_parquet('{processed_file}')
    WHERE TRY_CAST(din_interrupcaocarga AS TIMESTAMP) >= CAST('{INICIO}' AS DATE)
      AND TRY_CAST(din_interrupcaocarga AS TIMESTAMP) <  CAST('{FIM}' AS DATE) + INTERVAL 1 DAY
"""

duckdb.sql(f"""
    SELECT SUM(n_perturbacoes) AS perturbacoes
    FROM read_parquet('{processed_file}')
    WHERE data BETWEEN '{INICIO}' AND '{FIM}'
""").df()

In [ ]:
periodo = f"""
    SELECT cod_perturbacao AS perturbacao, id_estado AS estado
    FROM read_csv_auto('{interruption_file}', delim=';', header=true)
    WHERE id_subsistema = 'S'
      AND TRY_CAST(din_interrupcaocarga AS TIMESTAMP) >= CAST('{INICIO}' AS DATE)
      AND TRY_CAST(din_interrupcaocarga AS TIMESTAMP) <  CAST('{FIM}' AS DATE) + INTERVAL 1 DAY
"""

duckdb.sql(f"""
    WITH periodo AS ({periodo})
    SELECT estado,
           COUNT(DISTINCT perturbacao) AS perturbacoes,
           ROUND(100.0 * COUNT(DISTINCT perturbacao) / (SELECT COUNT(DISTINCT perturbacao) FROM periodo), 1) AS pct_perturbacoes
    FROM periodo
    GROUP BY estado
    ORDER BY perturbacoes DESC
""").df()